# Agentic RAG Pipeline for CleanTech

Multi-tool LangChain agent combining ChromaDB vector retrieval, Semantic Scholar paper search, and chain-of-thought reasoning over cleantech content. Evaluates base vs extended agent on BLEU and ROUGE metrics.

### Installing packages and importing Libraries

In [2]:
!pip install -q pandas numpy tqdm requests chromadb langchain langchain-community langchain-openai sentence-transformers evaluate bert-score rouge_score scikit-learn

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [31]:
# importing all required libraries for vector DB, agents, tools, and LLM
import os
import ast
import re
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import requests
import evaluate
import json

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_openai import ChatOpenAI
from langchain.tools import Tool
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain import hub

### Configuring global settings

In [32]:
# Setting my OpenAI key for the language model
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
# Creating the main language model that is driving all agent reasoning
language_model = ChatOpenAI(model="gpt-4o-mini",temperature=0.0)
# Defining file paths for the Cleantech article dataset and evaluation file
cleantech_articles_path = "/content/cleantech_media_dataset_v3_2024-10-28.csv"
cleantech_rag_eval_path = "/content/cleantech_rag_evaluation_data_2024-09-20.csv"
cleantech_50_answers_path = "/content/CleanTech-50answers.txt"
# Choosing a local directory to store the Chroma vector database
chroma_storage_path = "/content/chroma_cleantech_db"

## Part 1 - Base Agent with Retriever and Summarizer

### Loading and cleaning dataset

In [33]:
# Loading the Cleantech media dataset from CSV it contains titles, dates, authors, content, domains, and URLs
cleantech_articles_df = pd.read_csv(cleantech_articles_path)
# This function is converting any string representation of a list into plain text
def convert_list_string_to_plain_text(value):
    if isinstance(value, str):
        try:
            parsed_value = ast.literal_eval(value)
            if isinstance(parsed_value, list):
                # Joining list elements into a single text block
                return " ".join(parsed_value)
        except Exception:
            # Keeping the original string when parsing fails
            return value
    # Converting any non string value to string to keep the column consistent
    return str(value)
# Applying the cleaning function to the content column.
cleantech_articles_df["content"] = cleantech_articles_df["content"].apply(
    convert_list_string_to_plain_text)

# Combining title, cleaned content, and domain into one field
# This combined field will be used as the main text for retrieval
cleantech_articles_df["combined_text"] = (cleantech_articles_df[["title", "content", "domain"]]
    .fillna("").agg(" ".join, axis=1))

# Printing a quick sample to confirm the structure of the cleaned dataset
print("Sample rows from the cleaned Cleantech dataset:")
print(cleantech_articles_df[["title", "domain", "combined_text"]].head())


Sample rows from the cleaned Cleantech dataset:
                                               title           domain  \
0          XPeng Delivered ~100,000 Vehicles In 2021    cleantechnica   
1      Green Hydrogen: Drop In Bucket Or Big Splash?    cleantechnica   
2  World’ s largest floating PV plant goes online...      pv-magazine   
3  Iran wants to deploy 10 GW of renewables over ...      pv-magazine   
4  Eastern Interconnection Power Grid Said ‘ Bein...  naturalgasintel   

                                       combined_text  
0  XPeng Delivered ~100,000 Vehicles In 2021 Chin...  
1  Green Hydrogen: Drop In Bucket Or Big Splash? ...  
2  World’ s largest floating PV plant goes online...  
3  Iran wants to deploy 10 GW of renewables over ...  
4  Eastern Interconnection Power Grid Said ‘ Bein...  


### Creating document chunks and building Chroma vector store

In [34]:
# Splitting long article text into smaller chunks
# Chunking is helping the embedding model capture local context better
text_chunk_splitter = RecursiveCharacterTextSplitter(chunk_size=900,chunk_overlap=200)
document_chunk_list = []
# Walking through each article and producing multiple chunked Documents
for row_index, row_data in tqdm(cleantech_articles_df.iterrows(),total=len(cleantech_articles_df)):
    full_article_text = row_data["combined_text"].strip()
    # Skipping empty rows so useless chunks are not created
    if not full_article_text:
        continue
    # Attaching metadata for tracing retrievals back to the original row
    chunk_metadata = {"row_index": int(row_index),
        "article_url": row_data["url"],"article_domain": row_data["domain"]}
    # Splitting the article into overlapping chunks
    for chunk_text in text_chunk_splitter.split_text(full_article_text):
        document_chunk_list.append(Document(page_content=chunk_text, metadata=chunk_metadata))

print("Total number of document chunks created:", len(document_chunk_list))
# Creating an embedding model for building the vector database
embedding_model = SentenceTransformerEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

# Building the Chroma vector store from all document chunks
vector_store = Chroma.from_documents(documents=document_chunk_list,
    embedding=embedding_model,persist_directory=chroma_storage_path)

# Preparing a retriever interface; retrieving top-k chunks per query
retriever_top_k = 5
cleantech_retriever = vector_store.as_retriever(
    search_kwargs={"k": retriever_top_k})

100%|██████████| 20111/20111 [00:21<00:00, 951.45it/s] 


Total number of document chunks created: 149504


### Base tools - Retriever and Summarizer

In [35]:
# Tool for retrieving relevant document chunks for a question
def cleantech_retriever_tool(question_input_text: str) -> str:
    # Getting the most relevant chunks from the vector store
    retrieved_documents = cleantech_retriever.get_relevant_documents(question_input_text)
    # Formatting each document chunk with a label for clarity
    formatted_document_blocks = []
    for doc_idx, single_document in enumerate(retrieved_documents, start=1):
        formatted_document_blocks.append(f"[DOCUMENT {doc_idx}]\n{single_document.page_content}")
    combined_context_for_agent = "\n\n".join(formatted_document_blocks)
    return combined_context_for_agent
retriever_tool = Tool(name="cleantech_retriever",func=cleantech_retriever_tool,
    description="Retrieves relevant text passages from the Cleantech article collection.")

# Tool for summarizing a long context into a shorter explanation
def summarizer_tool(context_input_text: str) -> str:
    summarization_prompt = (
        "Summarize the following text in a clear and concise way. "
        "Focus on the main ideas and avoid repeating information.\n\n"
        f"Text:\n{context_input_text}\n\nSummary:")
    summarization_response = language_model.invoke(summarization_prompt)
    summarized_text_output = summarization_response.content
    return summarized_text_output
summarizer_tool_object = Tool(name="summarize_text",func=summarizer_tool,
    description="Summarizes retrieved Cleantech passages into a shorter explanation.")

sample_question_for_demo = "How is the global solar manufacturing supply chain changing after 2023?"
print("\nRetriever output for a sample question")
sample_context_text = cleantech_retriever_tool(sample_question_for_demo)
print(sample_context_text[:1200], "\n...")  # printing first part to keep it readable
print("\nSummarizer output for the retrieved context")
sample_summary_text = summarizer_tool(sample_context_text)
print(sample_summary_text)


Retriever output for a sample question
[DOCUMENT 1]
growth, whilst community solar is set to more than double by 2027. Looking directly to 2023, the report said that supply chain constraints are set to ease as more modules find their way through customs barriers and growth resumes. “ While 2022 was a tough year for the solar industry, we do expect some of the supply chain issues to ease, propelling 2023 growth to 41%, ” said Michelle Davis, principal analyst at Wood Mackenzie. A report last year from WoodMac already predicted that the IRA’ s true benefits won’ t be felt until 2024, hampered as it has been in its early stages by logistical processes and supply chain clogs. Similarly, domestic manufacturing capacity is set to grow massively, but not without a significant time lag which will require bolstering from imports to meet demand. The report called 2022 “ one of the most volatile policy years in industry history ”. Whilst the US

[DOCUMENT 2]
growth, whilst community solar is set

### Base agent

In [38]:
# Loading a generic tools-agent prompt template from LangChain Hub
base_agent_prompt_template = hub.pull("hwchase17/openai-tools-agent")

# Creating an OpenAI tools agent that knows how to call both tools
base_agent = create_openai_tools_agent(
    llm=language_model,
    tools=[retriever_tool, summarizer_tool_object],
    prompt=base_agent_prompt_template)

# Wrapping the agent in an executor for easy invocation
base_agent_executor = AgentExecutor(
    agent=base_agent,
    tools=[retriever_tool, summarizer_tool_object],
    verbose=False)  # Keeping output clean by disabling internal traces
print("\n" + "="*80)
print("PART 1: BASE AGENT DEMONSTRATION")
print("="*80)
# Sample questions for Part 1
part1_questions = [
    "How are rising material costs affecting wind turbine project timelines?",
    "What innovations in solar panel technology have emerged in 2024?",
    "How do energy storage systems improve grid stability?"]
print(f"\nTesting base agent with {len(part1_questions)} sample questions:")
for i, question in enumerate(part1_questions, 1):
    print(f"\n{'='*80}")
    print(f"QUESTION {i}:")
    print(f"{'='*80}")
    print(question)
    print("\nBASE AGENT ANSWER:")
    try:
        response = base_agent_executor.invoke({"input": question})
        print(response["output"])
    except Exception as e:
        print(f"Error: {e}")


PART 1: BASE AGENT DEMONSTRATION

Testing base agent with 3 sample questions:

QUESTION 1:
How are rising material costs affecting wind turbine project timelines?

BASE AGENT ANSWER:
Rising material costs are significantly affecting wind turbine project timelines. In the EU, for instance, the need to build 31GW of new wind turbines annually to meet 2030 targets is hindered by a drop in investments, which fell to €17bn last year—the lowest since 2009—allowing for only 10GW of construction. The costs of raw materials and shipping have surged, particularly due to geopolitical issues and inflation, resulting in a 40% increase in production costs over the last two years. This has led to fewer turbine orders and delays in new wind energy projects, ultimately impacting the timelines for these initiatives.

QUESTION 2:
What innovations in solar panel technology have emerged in 2024?

BASE AGENT ANSWER:
In 2024, several innovations in solar panel technology have emerged, focusing on enhancing 

## Part 2 - Extending Agent with custom tools (Semantic Scholar Search Tool + Reasoning Tool)

### Defining semantic scholar search tool and reasoning tool

In [39]:
# Defining a tool that is searching Semantic Scholar for research papers
def semantic_scholar_search_tool(query_text: str) -> str:
    try: # Setting the API endpoint for Semantic Scholar search
        api_url = "https://api.semanticscholar.org/graph/v1/paper/search"
        params = {  # Preparing the query parameters for the search request
            "query": query_text,
            "limit": 3,
            "fields": "title,abstract,year,authors"}
        response = requests.get(api_url, params=params)# Sending a GET request to the Semantic Scholar API
        data = response.json()
    except Exception as e:
        return f"Semantic Scholar request failed: {e}"
    if "data" not in data or len(data["data"]) == 0:
        return "No relevant research papers found in Semantic Scholar."

    paper_blocks = []
    # Looping through each returned paper entry
    for index, paper in enumerate(data["data"], start=1):
        title = paper.get("title", "No title available")   # Getting the paper title
        abstract = paper.get("abstract", "No abstract available.")  # Getting the abstract text
        year = paper.get("year", "Unknown year")
        paper_blocks.append(
            f"Paper {index}\nTitle: {title}\nYear: {year}\nAbstract: {abstract}")
    return "\n\n".join(paper_blocks)
# Creating a LangChain tool object for Semantic Scholar search
semantic_scholar_tool = Tool(name="semantic_scholar_search",func=semantic_scholar_search_tool,
    description="Searches Semantic Scholar for scientific papers related to a Cleantech question.")

# Defining a reasoning tool that is explaining text step by step
def reasoning_tool_function(content_to_explain: str) -> str:
  # Creating a prompt that is asking the model to explain the content clearly
    reasoning_prompt = ("Explain the following content step by step. "
        "Highlight the main points, trade-offs, and key implications.\n\n"
        f"Content:\n{content_to_explain}\n\nExplanation:")
    # Sending the explanation request to the language model
    reasoning_response = language_model.invoke(reasoning_prompt)
    reasoning_output_text = reasoning_response.content  # Extracting the explanation text
    return reasoning_output_text

reasoning_tool = Tool(name="reasoning_explainer",func=reasoning_tool_function,
    description="Provides a step-by-step explanation for Cleantech-related content.")

sample_research_query = "long duration energy storage"
print("\nSemantic Scholar search result for a sample research query")
semantic_demo_output = semantic_scholar_search_tool(sample_research_query)
print(semantic_demo_output)

print("\nReasoning tool output on a short piece of text ")
reasoning_demo_input = "Heat pumps reduce emissions by replacing natural gas furnaces but need high upfront installation costs."
reasoning_demo_output = reasoning_tool_function(reasoning_demo_input)
print(reasoning_demo_output)


Semantic Scholar search result for a sample research query
Paper 1
Title: Long Duration Energy Storage Using Hydrogen in Metal–Organic Frameworks: Opportunities and Challenges
Year: 2024
Abstract: Materials-based H2 storage plays a critical role in facilitating H2 as a low-carbon energy carrier, but there remains limited guidance on the technical performance necessary for specific applications. Metal–organic framework (MOF) adsorbents have shown potential in power applications, but need to demonstrate economic promises against incumbent compressed H2 storage. Herein, we evaluate the potential impact of material properties, charge/discharge patterns, and propose targets for MOFs’ deployment in long-duration energy storage applications including backup, load optimization, and hybrid power. We find that state-of-the-art MOF could outperform cryogenic storage and 350 bar compressed storage in applications requiring ≤8 cycles per year, but need ≥5 g/L increase in uptake to be cost-competit

## Creating advanced agent

In [40]:
# Collecting all tools in a single list for the advanced agent
advanced_tools_list = [retriever_tool,summarizer_tool_object,
    semantic_scholar_tool,reasoning_tool]
# Creating a more detailed agent prompt with required placeholders
advanced_agent_prompt_template = ChatPromptTemplate.from_messages([
    ("system",
    """You are an agent with access to multiple tools.

When answering:
1. First retrieve relevant cleantech articles using the retriever tool.
2. Summarize retrieved content using the summarizer tool.
3. If the question asks about technology, research trends, or scientific background, call the Semantic Scholar tool.
4. When reasoning through trade-offs or implications, call the reasoning tool.

Always:
- Provide a clear multi-paragraph answer.
- Include 4–6 key points.
- Include technical details when available.
- Avoid generic or overly short answers.

Return only the final answer to the user."""
    ),

    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Creating the advanced tools-based agent that can call all custom tools
advanced_agent = create_openai_tools_agent(llm=language_model,
    tools=advanced_tools_list,prompt=advanced_agent_prompt_template)
# Wrapping the advanced agent with an executor to run the workflow
advanced_agent_executor = AgentExecutor(agent=advanced_agent,
    tools=advanced_tools_list,verbose=False)
print("\n" + "="*80)
print("PART 2: ADVANCED AGENT DEMONSTRATION")
print("="*80)
# Load the evaluation dataset
eval_data_df = pd.read_csv(cleantech_rag_eval_path, sep=';')
print(f"Loaded evaluation dataset: {len(eval_data_df)} examples")

# Getting ALL questions from the datase
part2_questions = eval_data_df[['question_id', 'question']]
print(f"Testing advanced agent with {len(part2_questions)} questions from dataset\n")

# Testing advanced agent with all questions
for idx, row in part2_questions.iterrows():
    q_id = row['question_id']
    question = row['question']

    print(f"\n{'='*80}")
    print(f"PART 2 - QUESTION {idx+1} (Question ID: {q_id}):")
    print(f"{'='*80}")
    print(question)
    print("ADVANCED AGENT ANSWER:")

    try:
        response = advanced_agent_executor.invoke({"input": question})
        print(response["output"])
    except Exception as e:
        print(f"Error: {e}")


PART 2: ADVANCED AGENT DEMONSTRATION
Loaded evaluation dataset: 23 examples
Testing advanced agent with 23 questions from dataset


PART 2 - QUESTION 1 (Question ID: 1):
What is the innovation behind Leclanché's new method to produce lithium-ion batteries?
ADVANCED AGENT ANSWER:
Leclanché has recently introduced an innovative method for producing lithium-ion batteries that significantly enhances efficiency and sustainability in manufacturing. This advancement positions the company as a leader in the energy storage sector, particularly in the context of the growing demand for electric vehicles (EVs) and renewable energy solutions. Here are the key aspects of this innovation:

1. **Enhanced Production Efficiency**: Leclanché's new production process streamlines the manufacturing of lithium-ion batteries, potentially reducing production times and costs. This efficiency is crucial as the demand for batteries continues to rise, especially in the EV market.

2. **Sustainability Focus**: The

## Part 3 - Full Evaluation (Retrieval Accuracy, Answer Accuracy, LLM-as-Judge)

### Parsing CleanTech-50answers.txt

In [41]:
# Opening the 50-question evaluation file
with open(cleantech_50_answers_path, "r", encoding="utf-8") as file_handle:
    cleantech_50_raw_text = file_handle.read()
# Creating a regex pattern to capture question, reference answer, and article IDs
evaluation_pattern = re.compile(
    r"\*\*(\d+)\. Query:\*\*\s*(.*?)\n\*\*Desired Output:\*\*\s*(.*?)\n\*\*Referenced Articles:\*\*\s*([0-9,\s]+)",
    re.DOTALL)
# Creating a list to store extracted rows
evaluation_rows = []
# Extracting each evaluation block using regex matches
for match in evaluation_pattern.finditer(cleantech_50_raw_text):
    question_id_value = int(match.group(1)) # Extracting question number
    question_input_text = match.group(2).strip()  # Extracting the question text
    expected_reference_output_text = match.group(3).strip()# Extracting the expected reference answer
    referenced_ids_raw = match.group(4).strip() # Extracting the raw comma-separated IDs
    referenced_article_ids = [int(x.strip()) for x in referenced_ids_raw.split(",") if x.strip()]
    evaluation_rows.append({
        "question_id": question_id_value,
        "question_input_text": question_input_text,
        "expected_reference_output_text": expected_reference_output_text,
        "referenced_article_ids": referenced_article_ids})
evaluation_50_df = pd.DataFrame(evaluation_rows).sort_values("question_id").reset_index(drop=True)

# Printing the number of parsed items
print("Parsed evaluation questions:", len(evaluation_50_df))
# Printing a sample of parsed rows
print(evaluation_50_df.head(3))

Parsed evaluation questions: 50
   question_id                                question_input_text  \
0            1  How are innovative business models and nationa...   
1            2  Compare the scale, primary purpose, and fundin...   
2            3  What role do government policies play in catal...   

                      expected_reference_output_text referenced_article_ids  
0  Two primary models are emerging to overcome ge...         [47488, 47487]  
1  The Kosice project and Diverso's model represe...         [47488, 47487]  
2  Government policies are a critical driver for ...         [47488, 47487]  


In [42]:
demo_eval_question = evaluation_50_df.loc[0, "question_input_text"]
print("Evaluation Question Sample:\n", demo_eval_question)

Evaluation Question Sample:
 How are innovative business models and national policy support helping to overcome the high upfront costs that have traditionally hindered the adoption of geothermal heating?


### Retrieval Metrics

In [43]:
# Setting top-k retrieval count
top_k_documents = 5

# Creating a function to compute SEMANTIC retrieval metrics
def compute_semantic_retrieval_metrics(question_text, reference_answer, top_k=top_k_documents):
    # Retrieving top-k relevant document chunks
    retrieved_docs = cleantech_retriever.get_relevant_documents(question_text)
    retrieved_docs = retrieved_docs[:top_k]

    # Get embedding for the reference answer
    ref_embedding = embedding_model.embed_query(reference_answer)

    # Get embeddings for each retrieved document
    doc_embeddings = [embedding_model.embed_query(doc.page_content) for doc in retrieved_docs]

    # Compute cosine similarity between reference and each retrieved doc
    similarities = [cosine_similarity([ref_embedding], [doc_emb])[0][0] for doc_emb in doc_embeddings]

    # Compute metrics
    avg_similarity = np.mean(similarities)
    max_similarity = np.max(similarities)

    # Compute nDCG
    dcg = sum([sim / np.log2(i + 2) for i, sim in enumerate(similarities)])
    ideal_dcg = sum([1.0 / np.log2(i + 2) for i in range(len(similarities))])
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0.0

    return {
        "avg_similarity": avg_similarity,
        "max_similarity": max_similarity,
        "ndcg_score": ndcg
    }

retrieval_metrics_records = []

# Computing metrics for each evaluation row
for _, eval_row in tqdm(evaluation_50_df.iterrows(), total=len(evaluation_50_df), desc="Computing retrieval metrics"):
    metrics = compute_semantic_retrieval_metrics(
        question_text=eval_row["question_input_text"],
        reference_answer=eval_row["expected_reference_output_text"])

    # Saving the computed metrics
    retrieval_metrics_records.append({
        "question_id": eval_row["question_id"],
        "avg_similarity": metrics["avg_similarity"],
        "max_similarity": metrics["max_similarity"],
        "ndcg_score": metrics["ndcg_score"]})

retrieval_metrics_df = pd.DataFrame(retrieval_metrics_records)

print("\n" + "="*80)
print("RETRIEVAL METRICS (Semantic Similarity Based)")
print("="*80)
print(f"Average Similarity:  {retrieval_metrics_df['avg_similarity'].mean():.4f}")
print(f"Max Similarity:      {retrieval_metrics_df['max_similarity'].mean():.4f}")
print(f"Normalized DCG:      {retrieval_metrics_df['ndcg_score'].mean():.4f}")
print("\nInterpretation:")
print("  0.5+ = Excellent retrieval")
print("  0.3-0.5 = Good retrieval")
print("  <0.3 = Needs improvement")
print("="*80 + "\n")

print("Sample retrieval scores:")
print(retrieval_metrics_df.head(3))


Computing retrieval metrics: 100%|██████████| 50/50 [00:02<00:00, 19.23it/s]


RETRIEVAL METRICS (Semantic Similarity Based)
Average Similarity:  0.6697
Max Similarity:      0.7094
Normalized DCG:      0.6719

Interpretation:
  0.5+ = Excellent retrieval
  0.3-0.5 = Good retrieval
  <0.3 = Needs improvement

Sample retrieval scores:
   question_id  avg_similarity  max_similarity  ndcg_score
0            1        0.649591        0.672439    0.643000
1            2        0.689537        0.733131    0.698954
2            3        0.654729        0.665011    0.652978


### Generating advanced-agent answers

In [44]:
generation_records = []
# Looping over every evaluation question
for _, eval_row in tqdm(evaluation_50_df.iterrows(), total=len(evaluation_50_df)):
    question_text = eval_row["question_input_text"]
    try:
        agent_response = advanced_agent_executor.invoke({"input": question_text})
        assistant_generated_output = agent_response["output"] # Extracting assistant's answer
        status_value = "Success"
    except Exception as error_value:
        assistant_generated_output = f"ERROR: {error_value}"
        status_value = "Failed"
    generation_records.append({"question_id": eval_row["question_id"],
        "question_input_text": question_text,
        "expected_reference_output_text": eval_row["expected_reference_output_text"],
        "assistant_generated_output": assistant_generated_output,
        "generation_status": status_value})

generation_results_df = pd.DataFrame(generation_records)
print(f"\nTotal evaluated: {len(generation_results_df)}")
print(f"Successful: {(generation_results_df['generation_status'] == 'Success').sum()}/50")

100%|██████████| 50/50 [30:12<00:00, 36.25s/it]


Total evaluated: 50
Successful: 50/50


### ROUGE and BERTScore

In [45]:
# Filtering only successful generations
successful_generations_df = generation_results_df[
    generation_results_df["generation_status"] == "Success"].copy()
assistant_answer_list = successful_generations_df["assistant_generated_output"].tolist()
reference_answer_list = successful_generations_df["expected_reference_output_text"].tolist()
rouge_metric = evaluate.load("rouge") # Loading ROUGE evaluator
bertscore_metric = evaluate.load("bertscore") # Loading BERTScore evaluator
rouge_score_dict = rouge_metric.compute( # Computing ROUGE scores
    predictions=assistant_answer_list,references=reference_answer_list)
bertscore_dict = bertscore_metric.compute( # Computing BERTScore scores
    predictions=assistant_answer_list,references=reference_answer_list,lang="en")

print("ROUGE-L:", rouge_score_dict["rougeL"])
print("Mean BERTScore F1:", np.mean(bertscore_dict["f1"]))

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE-L: 0.1351398030545478
Mean BERTScore F1: 0.8465784370899201


### LLM-as-Judge

In [46]:
# Creating a function to evaluate an answer using the judge model
def judge_answer_quality(question_text, reference_text, model_answer):
  # Creating the evaluation prompt for the judge model
    judge_prompt = f"""
You are evaluating the accuracy of an AI-generated answer.
QUESTION:
{question_text}
REFERENCE ANSWER:
{reference_text}
ASSISTANT ANSWER:
{model_answer}

Give a score 1–5:
5 = Completely correct
4 = Mostly correct
3 = Partially correct
2 = Mostly incorrect
1 = Wrong or irrelevant

Return only JSON:
{{"score": number, "reason": "short explanation"}}
"""
    judge_response = language_model.invoke(judge_prompt).content.strip()# Getting the judge model output
    try:
        parsed = json.loads(judge_response) # Trying to parse the JSON returned by the model
        score_value = int(parsed.get("score", 3))
        reason_text = parsed.get("reason", "")
    except:
        score_value = 3
        reason_text = f"JSON parse error — raw output: {judge_response}"
    return score_value, reason_text

judge_records = []
for _, row in successful_generations_df.iterrows():  # Judging only the successful answers
    score_value, reason_text = judge_answer_quality(  # Running judge model for each answer
        row["question_input_text"],row["expected_reference_output_text"],
        row["assistant_generated_output"])
    judge_records.append({
        "question_id": row["question_id"],
        "judge_score": score_value,
        "judge_reason": reason_text})

judge_results_df = pd.DataFrame(judge_records)
print("Mean LLM-as-Judge score:", judge_results_df["judge_score"].mean())
print("\nSample judge outputs:")
print(judge_results_df.head(3))

Mean LLM-as-Judge score: 4.3

Sample judge outputs:
   question_id  judge_score                                       judge_reason
0            1            4  The assistant answer provides a comprehensive ...
1            2            4  The answer accurately compares the scale, purp...
2            3            4  The answer accurately highlights the role of g...


### Merging and printing final evaluation table

In [47]:
# Merging evaluation set with retrieval results
final_evaluation_df = (evaluation_50_df.merge(retrieval_metrics_df, on="question_id", how="left")
    .merge(generation_results_df, on="question_id", how="left")
    .merge(judge_results_df, on="question_id", how="left"))

print("\n" + "="*80)
print("PART 3: COMPLETE EVALUATION - ALL 50 QUESTIONS WITH METRICS")
print("="*80)
print(f"Total Questions: {len(final_evaluation_df)}")
print(f"Successful: {(final_evaluation_df['generation_status'] == 'Success').sum()}/50")
print("="*80 + "\n")

# Display all 50 questions with their metrics
for i in range(len(final_evaluation_df)):
    row = final_evaluation_df.iloc[i]

    print(f"\n{'='*80}")
    print(f"QUESTION {i+1}/50 - ID: {row['question_id']}")
    print(f"{'='*80}")

    print(f"\n{'-'*80}")
    print("QUESTION:")
    print(f"{'-'*80}")
    print(row['question_input_text_x'])

    print(f"\n{'-'*80}")
    print("REFERENCE ANSWER:")
    print(f"{'-'*80}")
    print(row['expected_reference_output_text_x'])

    print(f"\n{'-'*80}")
    print("GENERATED ANSWER:")
    print(f"{'-'*80}")
    print(row['assistant_generated_output'])

    print(f"\n{'-'*80}")
    print("METRICS:")
    print(f"{'-'*80}")
    print(f"Retrieval - Avg Similarity: {row['avg_similarity']:.4f}")
    print(f"Retrieval - Max Similarity: {row['max_similarity']:.4f}")
    print(f"Retrieval - nDCG: {row['ndcg_score']:.4f}")

    if pd.notna(row['judge_score']):
        print(f"LLM Judge - Score: {int(row['judge_score'])}/5")
        print(f"LLM Judge - Reason: {row['judge_reason']}")


PART 3: COMPLETE EVALUATION - ALL 50 QUESTIONS WITH METRICS
Total Questions: 50
Successful: 50/50


QUESTION 1/50 - ID: 1

--------------------------------------------------------------------------------
QUESTION:
--------------------------------------------------------------------------------
How are innovative business models and national policy support helping to overcome the high upfront costs that have traditionally hindered the adoption of geothermal heating?

--------------------------------------------------------------------------------
REFERENCE ANSWER:
--------------------------------------------------------------------------------
Two primary models are emerging to overcome geothermal's high capital costs. In Slovakia, the Kosice geothermal project is being funded as a national priority through the Just Transformation Fund, receiving EUR 56.1 million. This state-level support treats geothermal as critical public infrastructure for energy security and reducing fossil fuel d